In [ ]:
import torch
import sys
sys.path.append("..")
from model.attention import Attention
from config import D_MODEL, NUM_HEADS

torch.manual_seed(42)

# Create attention module
attention = Attention(d_model=D_MODEL,num_heads=NUM_HEADS,)

# Test normal causal self-attention
B, T = 2, 4
x = torch.randn(B, T, D_MODEL)

output = attention(x)

print("Input shape: ", x.shape)
print("Output shape:", output.shape)
assert output.shape == x.shape

# Inspect attention dimensions
q = attention.q_proj(x).reshape(B, T, NUM_HEADS, attention.head_dim)
q = q.transpose(1, 2)

k = attention.k_proj(x).reshape(B, T, NUM_HEADS, attention.head_dim)
k = k.transpose(1, 2)

v = attention.v_proj(x).reshape(B, T, NUM_HEADS, attention.head_dim)
v = v.transpose(1, 2)

print("\nQ shape:", q.shape)
print("K shape:", k.shape)
print("V shape:", v.shape)
print("Head dimension:", attention.head_dim)

assert q.shape == (B, NUM_HEADS, T, attention.head_dim)
assert k.shape == (B, NUM_HEADS, T, attention.head_dim)
assert v.shape == (B, NUM_HEADS, T, attention.head_dim)

# Verify causal behavior: changing a future token must not change earlier outputs
x_modified = x.clone()
x_modified[:, -1, :] = torch.randn(B, D_MODEL)

output_original = attention(x)
output_modified = attention(x_modified)

print(
    "\nEarlier positions unchanged:",
    torch.allclose(
        output_original[:, :-1],
        output_modified[:, :-1],
        atol=1e-6,
    ),
)

assert torch.allclose(
    output_original[:, :-1],
    output_modified[:, :-1],
    atol=1e-6,
)

# Verify KV cache
prefix = x[:, :3, :]
next_token = x[:, 3:, :]

prefix_output, cache = attention(
    prefix,
    use_cache=True,
)

cached_output, updated_cache = attention(
    next_token,
    past_key_value=cache,
    use_cache=True,
)

full_output = attention(x)

print("\nPrefix output shape:", prefix_output.shape)
print("Cached output shape:", cached_output.shape)
print("Full output shape:", full_output.shape)

print("Cached K shape:", cache[0].shape)
print("Cached V shape:", cache[1].shape)

# Cached next-token output should match the full-sequence result
print(
    "KV cache matches full attention:",
    torch.allclose(
        cached_output,
        full_output[:, -1:, :],
        atol=1e-5,
    ),
)

assert torch.allclose(
    cached_output,
    full_output[:, -1:, :],
    atol=1e-5,
)

print("\nAll attention tests passed.")

Input shape:  torch.Size([2, 4, 512])
Output shape: torch.Size([2, 4, 512])

Q shape: torch.Size([2, 8, 4, 64])
K shape: torch.Size([2, 8, 4, 64])
V shape: torch.Size([2, 8, 4, 64])
Head dimension: 64

Earlier positions unchanged: True

Prefix output shape: torch.Size([2, 3, 512])
Cached output shape: torch.Size([2, 1, 512])
Full output shape: torch.Size([2, 4, 512])
Cached K shape: torch.Size([2, 8, 3, 64])
Cached V shape: torch.Size([2, 8, 3, 64])
KV cache matches full attention: True

All attention tests passed.
